# Autogen (handoff)

### Setup

In [ ]:
import datetime
import os
import sqlite3
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, TypedDict, cast

from autogen_ext.models.openai import AzureOpenAIChatCompletionClient
from azure.core.credentials_async import AsyncTokenCredential
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.search.documents.aio import AsyncSearchItemPaged, SearchClient
from azure.search.documents.models import (
    QueryCaptionResult,
    QueryType,
    VectorizedQuery,
    VectorQuery,
)
from dotenv import load_dotenv
from openai import AsyncAzureOpenAI
from openai.types.chat import ChatCompletion
from openai_messages_token_helper import build_messages, get_token_limit

### openai connection

In [97]:
# AZURE_OPENAI_CHATGPT_DEPLOYMENT="chat"
# AZURE_OPENAI_API_VERSION="2024-10-21"
# AZURE_OPENAI_CHATGPT_MODEL="gpt-4o-mini"
# AZURE_OPENAI_SERVICE="cog-5ivxbh72auo44-000"
load_dotenv(dotenv_path=r"..\.env")

OPENAI_HOST = os.getenv("OPENAI_HOST", "azure")
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHATGPT_DEPLOYMENT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_CHATGPT_MODEL = os.getenv("AZURE_OPENAI_CHATGPT_MODEL")
AZURE_OPENAI_SERVICE = os.getenv("AZURE_OPENAI_SERVICE")

OPENAI_EMB_MODEL = os.getenv("AZURE_OPENAI_EMB_MODEL_NAME", "text-embedding-ada-002")
OPENAI_EMB_DIMENSIONS = int(os.getenv("AZURE_OPENAI_EMB_DIMENSIONS", 1536))
AZURE_OPENAI_EMB_DEPLOYMENT = (
    os.getenv("AZURE_OPENAI_EMB_DEPLOYMENT")
    if OPENAI_HOST.startswith("azure")
    else None
)

AZURE_SEARCH_SERVICE = os.environ["AZURE_SEARCH_SERVICE"]
AZURE_SEARCH_INDEX = os.environ["AZURE_SEARCH_INDEX"]
AZURE_SEARCH_ENDPOINT = f"https://{AZURE_SEARCH_SERVICE}.search.windows.net"
AZURE_SEARCH_QUERY_LANGUAGE = os.getenv("AZURE_SEARCH_QUERY_LANGUAGE", "en-us")
AZURE_SEARCH_QUERY_SPELLER = os.getenv("AZURE_SEARCH_QUERY_SPELLER", "lexicon")

CHATGPT_TOKEN_LIMIT = get_token_limit(AZURE_OPENAI_CHATGPT_MODEL)

In [ ]:
azure_credential = DefaultAzureCredential(exclude_shared_token_cache_credential=True)

token_provider = get_bearer_token_provider(
    azure_credential, "https://cognitiveservices.azure.com/.default"
)

autogen_openai_client = AzureOpenAIChatCompletionClient(
    azure_deployment=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model=AZURE_OPENAI_CHATGPT_MODEL,
    api_version=AZURE_OPENAI_API_VERSION,
    azure_endpoint=f"https://{AZURE_OPENAI_SERVICE}.openai.azure.com",
    azure_ad_token_provider=token_provider,
)

openai_client = AsyncAzureOpenAI(
    api_version=AZURE_OPENAI_API_VERSION,
    azure_endpoint=f"https://{AZURE_OPENAI_SERVICE}.openai.azure.com",
    azure_ad_token_provider=token_provider,
)

In [5]:
# # testing openai connection
# from autogen_agentchat.agents import AssistantAgent
# from autogen_agentchat.ui import Console

# async def get_weather(city: str) -> str:
#     """Get the weather for a given city."""
#     return f"The weather in {city} is 73 degrees and Sunny."


# # Define an AssistantAgent with the model, tool, system message, and reflection enabled.
# # The system message instructs the agent via natural language.
# agent = AssistantAgent(
#     name="weather_agent",
#     model_client=autogen_openai_client,
#     tools=[get_weather],
#     system_message="You are a helpful assistant.",
#     reflect_on_tool_use=True,
#     model_client_stream=True,  # Enable streaming tokens from the model client.
# )


# # Run the agent and stream the messages to the console.
# async def main() -> None:
#     await Console(agent.run_stream(task="What is the weather in New York?"))


# # NOTE: if running this inside a Python script you'll need to use asyncio.run(main()).
# await main()

---------- user ----------
What is the weather in New York?
---------- weather_agent ----------
[FunctionCall(id='call_3wGg1RmOUQu5UpWpLbWDgUIG', arguments='{"city":"New York"}', name='get_weather')]
---------- weather_agent ----------
[FunctionExecutionResult(content='The weather in New York is 73 degrees and Sunny.', name='get_weather', call_id='call_3wGg1RmOUQu5UpWpLbWDgUIG', is_error=False)]
---------- weather_agent ----------
The weather in New York is currently 73 degrees and sunny.


### Parameters

In [ ]:
SEARCH_MAX_RESULTS = 10
TEMPERATURE = 0.0
SEED = 1234
USE_TEXT_SEARCH = "hybrid"
USE_VECTOR_SEARCH = "hybrid"
USE_SEMANTIC_RANKER = True
USE_SEMANTIC_CAPTIONS = False
MINIMUM_SEARCH_SCORE = 0.022
MINIMUM_RERANKER_SCORE = 2.21
COSINE_SIMILARITY_THRESHOLD = 0.90
VECTOR_WEIGHT = 1.75

RESPONSE_TOKEN_LIMIT = 512
CHATGPT_TOKEN_LIMIT = 128000

### Define Azure search functions

In [ ]:
@dataclass
class Document:
    id: Optional[str]
    parent_id: Optional[str]
    title: Optional[str]
    pr_name: Optional[str]
    date_modified: Optional[str]
    cover_image_url: Optional[str]
    full_url: Optional[str]
    content_category: Optional[str]
    chunks: Optional[str]
    embedding: Optional[list[float]]
    captions: list[QueryCaptionResult]
    score: Optional[float] = None
    reranker_score: Optional[float] = None

    def serialize_for_results(self) -> dict[str, Any]:
        return {
            "id": self.id,
            "parent_id": self.parent_id,
            "title": self.title,
            "cover_image_url": self.cover_image_url,
            "full_url": self.full_url,
            "content_category": self.content_category,
            "chunks": self.chunks,
            "embedding": Document.trim_embedding(self.embedding),
            "captions": (
                [
                    {
                        "additional_properties": caption.additional_properties,
                        "text": caption.text,
                        "highlights": caption.highlights,
                    }
                    for caption in self.captions
                ]
                if self.captions
                else []
            ),
            "score": self.score,
            "reranker_score": self.reranker_score,
        }

    @classmethod
    def trim_embedding(cls, embedding: Optional[list[float]]) -> Optional[list[float]]:
        """Returns a trimmed list of floats from the vector embedding."""
        if embedding:
            if len(embedding) > 2:
                # Format the embedding list to show the first 2 items followed by the count of the remaining items."""
                return f"[{embedding[0]}, {embedding[1]} ...+{len(embedding) - 2} more]"
            else:
                return str(embedding)

        return None

In [ ]:
async def compute_text_embedding(q: str):
    SUPPORTED_DIMENSIONS_MODEL = {
        "text-embedding-ada-002": False,
        "text-embedding-3-small": True,
        "text-embedding-3-large": True,
    }

    class ExtraArgs(TypedDict, total=False):
        dimensions: int

    dimensions_args: ExtraArgs = (
        {"dimensions": OPENAI_EMB_DIMENSIONS}
        if SUPPORTED_DIMENSIONS_MODEL[OPENAI_EMB_MODEL]
        else {}
    )
    embedding = await openai_client.embeddings.create(  # noqa: F704
        # Azure OpenAI takes the deployment name as the model name
        model=(
            AZURE_OPENAI_EMB_DEPLOYMENT
            if AZURE_OPENAI_EMB_DEPLOYMENT
            else OPENAI_EMB_MODEL
        ),
        input=q,
        **dimensions_args,
    )
    query_vector = embedding.data[0].embedding
    return VectorizedQuery(
        vector=query_vector, k_nearest_neighbors=50, fields="embedding"
    )

In [ ]:
async def perform_search_and_fetch_documents(
    endpoint: str,
    index_name: str,
    azure_credential: AsyncTokenCredential,
    query_text: str,
    vectors: list[VectorQuery],
    usetextsearch: bool,
    usevectorsearch: bool,
    usesemanticranker: bool,
    top: int,
    usesemanticcaptions: bool,
    filter: str,
    # query_language: str,
    # query_speller: str,
) -> list[Document]:

    async with SearchClient(
        endpoint=endpoint,
        index_name=index_name,
        credential=azure_credential,
    ) as search_client:
        search_text = query_text if usetextsearch else ""
        search_vectors = vectors if usevectorsearch else []
        if usesemanticranker:
            results: AsyncSearchItemPaged = await search_client.search(
                search_text=search_text,
                filter=filter,
                top=top,
                query_caption=(
                    "extractive|highlight-false" if usesemanticcaptions else None
                ),
                vector_queries=search_vectors,
                query_type=QueryType.SEMANTIC,
                # query_language=query_language,
                # query_speller=query_speller,
                semantic_configuration_name="default",
                semantic_query=query_text,
            )
        else:
            results: AsyncSearchItemPaged = await search_client.search(
                search_text=search_text,
                filter=filter,
                top=top,
                vector_queries=search_vectors,
            )

        documents = []
        async for page in results.by_page():
            async for document in page:
                documents.append(
                    Document(
                        id=document.get("id"),
                        parent_id=document.get("parent_id"),
                        title=document.get("title"),
                        pr_name=document.get("pr_name"),
                        date_modified=document.get("date_modified"),
                        cover_image_url=document.get("cover_image_url"),
                        full_url=document.get("full_url"),
                        content_category=document.get("content_category"),
                        chunks=document.get("chunks"),
                        embedding=document.get("embedding"),
                        captions=cast(
                            list[QueryCaptionResult], document.get("@search.captions")
                        ),
                        score=document.get("@search.score"),
                        reranker_score=document.get("@search.reranker_score"),
                    )
                )

        return documents


def get_citation(sourcepage: str, use_image_citation: bool) -> str:
    if use_image_citation:
        return sourcepage
    else:
        path, ext = os.path.splitext(sourcepage)
        if ext.lower() == ".png":
            page_idx = path.rfind("-")
            page_number = int(path[page_idx + 1 :])
            return f"{path[:page_idx]}.pdf#page={page_number}"

        return sourcepage


def nonewlines(s: str) -> str:
    return s.replace("\n", " ").replace("\r", " ")


async def get_sources_content(
    results: list[Document], use_semantic_captions: bool, use_image_citation: bool
) -> list[str]:
    if use_semantic_captions:
        return [
            {
                "index_id": doc.id or "",  # noqa: F821
                "article_id": doc.parent_id or "",  # noqa: F821
                "title": doc.title or "",  # noqa: F821
                "pr_name": doc.pr_name or "",  # noqa: F821
                "url": get_citation(
                    (doc.full_url or ""), use_image_citation
                )  # noqa: F821
                + " ["
                + (doc.date_modified or "no date")  # noqa: F821
                + "]",
                "captions": nonewlines(
                    " . ".join([cast(str, c.text) for c in (doc["captions"] or [])])
                ),  # noqa: F821
                "search_score": doc.score or "",
                "reranker_score": doc.reranker_score or "",
                "embedding": doc.embedding or "",
            }
            for doc in results
        ]
    else:
        return [
            {
                "index_id": doc.id or "",
                "article_id": doc.parent_id or "",
                "title": doc.title or "",
                "pr_name": doc.pr_name or "",
                "url": get_citation((doc.full_url or ""), use_image_citation)
                + " ["
                + (doc.date_modified or "no date")
                + "]",
                "chunk": (get_citation((doc.title or ""), use_image_citation))
                + ": "
                + nonewlines(doc.chunks or ""),
                "search_score": doc.score or "",
                "reranker_score": doc.reranker_score or "",
                "embedding": doc.embedding or "",
            }
            for doc in results
        ]

### Define tool calls

In [ ]:
def fetch_vaccination_history(user_id: int) -> List[Dict]:
    """Retrieve user's past vaccinations from the database."""
    conn = sqlite3.connect(r"..\data\vaccination_db.sqlite")
    cursor = conn.cursor()
    # sql query to filtered by user_id in descending vaccination date
    query = """
        SELECT vr.record_id, v.vaccine_name, vr.dose_number, vr.vaccination_date, p.polyclinic_name 
        FROM VaccineRecords vr
        JOIN Vaccines v ON vr.vaccine_id = v.vaccine_id
        JOIN Polyclinics p ON vr.polyclinic_id = p.polyclinic_id
        WHERE vr.user_id = ?
        ORDER BY vr.vaccination_date DESC;
    """
    cursor.execute(query, (user_id,))
    records = cursor.fetchall()
    conn.close()
    vaccination_history_records = [
        {
            "record_id": r[0],
            "vaccine_name": r[1],
            "dose_number": r[2],
            "date": r[3],
            "polyclinic": r[4],
        }
        for r in records
    ]
    return vaccination_history_records

In [6]:
def fetch_user_profile(user_id: int) -> Optional[Dict]:
    """Retrieve user profile details."""
    conn = sqlite3.connect(r"..\data\vaccination_db.sqlite")
    cursor = conn.cursor()
    query = "SELECT user_id, name, email, date_of_birth FROM Users WHERE user_id = ?;"
    cursor.execute(query, (user_id,))
    user = cursor.fetchone()
    conn.close()
    if user:
        return {
            "user_id": user[0],
            "name": user[1],
            "email": user[2],
            "date_of_birth": user[3],
        }
    return None

In [42]:
def check_available_slots(vaccine_name: str, polyclinic_name: str) -> List[Dict]:
    """Find available slots for a specific vaccine."""
    conn = sqlite3.connect(r"..\data\vaccination_db.sqlite")
    cursor = conn.cursor()
    query = """
        SELECT bs.slot_id, p.polyclinic_name, bs.slot_datetime 
        FROM BookingSlots bs
        JOIN Polyclinics p ON bs.polyclinic_id = p.polyclinic_id
        JOIN Vaccines v ON bs.vaccine_id = v.vaccine_id
        WHERE v.vaccine_name = ? AND bs.is_booked = 0
    """
    params = [vaccine_name]
    query += " AND p.polyclinic_name LIKE ?"
    params.append(f"%{polyclinic_name}%")
    query += " ORDER BY bs.slot_datetime ASC;"
    cursor.execute(query, tuple(params))
    slots = cursor.fetchall()
    conn.close()
    return [{"slot_id": s[0], "polyclinic": s[1], "datetime": s[2]} for s in slots]

In [ ]:
def book_appointment(
    slot_id: int, user_id: int, action: str, new_slot_id: int = None
) -> str:
    """Handles booking, rescheduling, or cancelling an appointment."""
    conn = sqlite3.connect(r"..\data\vaccination_db.sqlite")
    cursor = conn.cursor()

    if action == "new":
        # Try to book a new appointment if the slot is available
        cursor.execute(
            "SELECT is_booked FROM BookingSlots WHERE slot_id = ?;", (slot_id,)
        )
        result = cursor.fetchone()

        if result and result[0] == 0:  # Slot is available
            cursor.execute(
                "UPDATE BookingSlots SET is_booked = 1 WHERE slot_id = ?;", (slot_id,)
            )
            # conn.commit()
            conn.close()
            return "Appointment booked successfully."
        conn.close()
        return "Slot is no longer available."

    elif action == "reschedule":
        # Cancel the old appointment (set is_booked to 0)
        cursor.execute(
            "SELECT is_booked FROM BookingSlots WHERE slot_id = ?;", (slot_id,)
        )
        result = cursor.fetchone()

        if result and result[0] == 1:  # Slot is booked
            cursor.execute(
                "UPDATE BookingSlots SET is_booked = 0 WHERE slot_id = ?;", (slot_id,)
            )
            # conn.commit()
            print(
                "Existing appointment has been cancelled. Rebooking new appointment slot.."
            )

        # Book the new slot if available
        cursor.execute(
            "SELECT is_booked FROM BookingSlots WHERE slot_id = ?;", (new_slot_id,)
        )
        result = cursor.fetchone()

        if result and result[0] == 0:  # Slot is available
            cursor.execute(
                "UPDATE BookingSlots SET is_booked = 1 WHERE slot_id = ?;",
                (new_slot_id,),
            )
            # conn.commit()
            conn.close()
            return "Appointment successfully rescheduled."
        conn.close()
        return "New slot is not available."

    elif action == "cancel":
        # Cancel the old appointment (set is_booked to 0)
        cursor.execute(
            "SELECT is_booked FROM BookingSlots WHERE slot_id = ?;", (slot_id,)
        )
        result = cursor.fetchone()

        if result and result[0] == 1:  # Slot is booked
            cursor.execute(
                "UPDATE BookingSlots SET is_booked = 0 WHERE slot_id = ?;", (slot_id,)
            )
            # conn.commit()
            conn.close()
            return "Appointment successfully cancelled."

        conn.close()
        return "No appointment found to cancel."

    conn.close()
    return "Invalid action."

In [ ]:
async def recommend_vaccines(user_id: int):
    """
    Recommend vaccines based on user profile and vaccination history.
    """
    # Step 1: Fetch user profile (age, gender)
    conn = sqlite3.connect(r"..\data\vaccination_db.sqlite")
    cursor = conn.cursor()
    cursor.execute("SELECT date_of_birth FROM Users WHERE user_id = ?", (user_id,))
    user = cursor.fetchone()
    if not user:
        return {"error": "User not found"}

    birth_date = datetime.datetime.strptime(user[0], "%Y-%m-%d").date()
    age = (datetime.date.today() - birth_date).days // 365

    # cursor.execute("SELECT gender FROM Users WHERE user_id = ?", (user_id,))
    # gender = cursor.fetchone()[0]
    gender = "male"  # missing gender info in data for now

    print(age, gender)

    # Step 2: Query Azure Index for recommended vaccines
    query_text = f"vaccination guidelines for a {age}-year-old {gender}"
    vectors = [await compute_text_embedding(query_text)]
    documents = await perform_search_and_fetch_documents(
        endpoint=AZURE_SEARCH_ENDPOINT,
        index_name=AZURE_SEARCH_INDEX,
        azure_credential=azure_credential,
        query_text=query_text,
        vectors=vectors,
        usetextsearch=USE_TEXT_SEARCH,
        usevectorsearch=USE_VECTOR_SEARCH,
        usesemanticranker=USE_SEMANTIC_RANKER,
        top=5,  # Get top 5 recommendations
        usesemanticcaptions=USE_SEMANTIC_CAPTIONS,
        filter="parent_id eq '1434610_content_js'",
        # filter=None
    )

    sources_content = await get_sources_content(documents, USE_SEMANTIC_CAPTIONS, False)
    content = " ".join(doc["chunk"] for doc in sources_content)

    vaccination_history_records = fetch_vaccination_history(user_id)

    # Step 3: Use LLM to recommend vaccinations
    vaccine_recommendation_prompt = f"""
    Given the following vaccination guidelines, list the recommended vaccines for a {age}-year-old {gender}.
    Exclude any vaccines that are irrelevant for the age group and vaccinations that user has already taken based on the vaccination history. Provide a brief purpose for each vaccine.

    Guidelines:
    {content}

    Vaccination history:
    {vaccination_history_records}

    Respond in a numbered bullet format of the recommended vaccine and its purpose.
    """

    messages = build_messages(
        model=AZURE_OPENAI_CHATGPT_MODEL,
        system_prompt=vaccine_recommendation_prompt,
        max_tokens=CHATGPT_TOKEN_LIMIT - RESPONSE_TOKEN_LIMIT,
    )

    chat_completion: ChatCompletion = await openai_client.chat.completions.create(
        # Azure OpenAI takes the deployment name as the model name
        model=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
        messages=messages,
        temperature=0,
        max_tokens=RESPONSE_TOKEN_LIMIT,
        n=1,
        stream=False,
        seed=SEED,
    )

    return chat_completion.choices[0].message.content

## End